In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

ratings = pd.read_csv("../data/ml-latest-small/ratings.csv")
filmes = pd.read_csv("../data/movies_enriched.csv")

titulos = filmes.set_index("movieId")["title"]
print(ratings.shape)

(100836, 4)


In [2]:
ratings["rating_c"] = ratings["rating"] - ratings.groupby("userId")["rating"].transform("mean")
ratings[["userId", "movieId", "rating", "rating_c"]].head()

,userId,movieId,rating,rating_c
0,1,1,4.0,-0.366379
1,1,3,4.0,-0.366379
2,1,6,4.0,-0.366379
3,1,47,5.0,0.633621
4,1,50,5.0,0.633621


In [15]:
usuarios = ratings["userId"].astype("category")
filmes_cat = ratings["movieId"].astype("category")

movie_ids = filmes_cat.cat.categories

matriz = csr_matrix(
    (ratings["rating_c"], (usuarios.cat.codes, filmes_cat.cat.codes))
)

item_matrix = matriz.T.tocsr()
print("Formato (filmes x usuários):", item_matrix.shape)

Formato (filmes x usuários): (9724, 610)


In [8]:
n_aval = (
    ratings.groupby("movieId")["rating"].count().reindex(movie_ids).values
)
posicao = pd.Series(np.arange(len(movie_ids)), index=movie_ids)

In [9]:
def buscar_id(trecho):
    achados = filmes[filmes["title"].str.contains(trecho, case=False, regex=False)]
    return achados[["movieId", "title"]]


def similares_colab(movie_id, n=10, min_avaliacoes=20):
    """Filmes mais parecidos pelo comportamento dos usuários."""
    i = posicao[movie_id]
    scores = cosine_similarity(item_matrix[i], item_matrix).flatten()

    scores[n_aval < min_avaliacoes] = -1
    scores[i] = -1 

    top = scores.argsort()[::-1][:n]
    ids = movie_ids[top]
    return pd.DataFrame({
        "movieId": ids,
        "title": titulos.loc[ids].values,
        "similaridade": scores[top].round(3),
        "n_avaliacoes": n_aval[top],
    })

In [10]:
buscar_id("Dark Knight")

,movieId,title
6710,58559,"Dark Knight, The (2008)"
7768,91529,"Dark Knight Rises, The (2012)"
8032,98124,"Batman: The Dark Knight Returns, Part 1 (2012)"
8080,99813,"Batman: The Dark Knight Returns, Part 2 (2013)"


In [11]:
similares_colab(58559) 

,movieId,title,similaridade,n_avaliacoes
0,91529,"Dark Knight Rises, The (2012)",0.470,76
1,112852,Guardians of the Galaxy (2014),0.403,59
2,33794,Batman Begins (2005),0.400,116
3,57669,In Bruges (2008),0.372,41
4,89745,"Avengers, The (2012)",0.362,69
5,60069,WALL·E (2008),0.358,104
6,59315,Iron Man (2008),0.346,94
7,48516,"Departed, The (2006)",0.342,107
8,68954,Up (2009),0.338,105
9,38061,Kiss Kiss Bang Bang (2005),0.335,35


In [12]:
buscar_id("Toy Story")

,movieId,title
0,1,Toy Story (1995)
2355,3114,Toy Story 2 (1999)
7355,78499,Toy Story 3 (2010)


In [13]:
similares_colab(1)

,movieId,title,similaridade,n_avaliacoes
0,3114,Toy Story 2 (1999),0.403,97
1,78499,Toy Story 3 (2010),0.327,55
2,588,Aladdin (1992),0.327,183
3,1148,Wallace & Gromit: The Wrong Trousers (1993),0.305,56
4,1270,Back to the Future (1985),0.277,171
5,8961,"Incredibles, The (2004)",0.275,125
6,3671,Blazing Saddles (1974),0.272,62
7,6377,Finding Nemo (2003),0.263,141
8,2716,Ghostbusters (a.k.a. Ghost Busters) (1984),0.250,120
9,1198,Raiders of the Lost Ark (Indiana Jones and the...,0.237,200


In [14]:
print("Com min 20:")
print(similares_colab(1, min_avaliacoes=20)[["title", "similaridade", "n_avaliacoes"]])

print("\nCom min 1:")
print(similares_colab(1, min_avaliacoes=1)[["title", "similaridade", "n_avaliacoes"]])

Com min 20:
                                               title  similaridade  \
0                                 Toy Story 2 (1999)         0.403   
1                                 Toy Story 3 (2010)         0.327   
2                                     Aladdin (1992)         0.327   
3        Wallace & Gromit: The Wrong Trousers (1993)         0.305   
4                          Back to the Future (1985)         0.277   
5                            Incredibles, The (2004)         0.275   
6                             Blazing Saddles (1974)         0.272   
7                                Finding Nemo (2003)         0.263   
8         Ghostbusters (a.k.a. Ghost Busters) (1984)         0.250   
9  Raiders of the Lost Ark (Indiana Jones and the...         0.237   

   n_avaliacoes  
0            97  
1            55  
2           183  
3            56  
4           171  
5           125  
6            62  
7           141  
8           120  
9           200  

Com min 1:
       

In [16]:
contagem = ratings.groupby("movieId")["rating"].count()
raros = contagem[(contagem >= 5) & (contagem <= 8)].index[:5]
titulos.loc[raros]

movieId
4     Waiting to Exhale (1995)
8          Tom and Huck (1995)
13                Balto (1995)
42      Dead Presidents (1995)
43          Restoration (1995)
Name: title, dtype: str

In [17]:
ID = raros[0]
similares_colab(ID, min_avaliacoes=1)

,movieId,title,similaridade,n_avaliacoes
0,46664,"Fallen Idol, The (1948)",0.457,1
1,2800,Little Nemo: Adventures in Slumberland (1992),0.457,1
2,7395,Cheaper by the Dozen (1950),0.457,2
3,3964,"Adventures of Ichabod and Mr. Toad, The (1949)",0.426,3
4,1551,Buddy (1997),0.419,3
5,6978,Slacker (1991),0.399,4
6,4814,Don't Say a Word (2001),0.389,5
7,72,Kicking and Screaming (1995),0.383,7
8,7255,Win a Date with Tad Hamilton! (2004),0.375,8
9,26614,"Bourne Identity, The (1988)",0.373,9


In [18]:
similares_colab(ID, min_avaliacoes=20)

,movieId,title,similaridade,n_avaliacoes
0,371,"Paper, The (1994)",0.363,21
1,56775,National Treasure: Book of Secrets (2007),0.252,24
2,327,Tank Girl (1995),0.247,34
3,224,Don Juan DeMarco (1995),0.245,40
4,3360,Hoosiers (a.k.a. Best Shot) (1986),0.244,22
5,762,Striptease (1996),0.238,41
6,524,Rudy (1993),0.233,34
7,3476,Jacob's Ladder (1990),0.223,26
8,53322,Ocean's Thirteen (2007),0.222,33
9,342,Muriel's Wedding (1994),0.215,33
